In [ ]:
# Project: Financial Inclusion under PMJDY
# Student: Nancy Singh

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

print("Libraries imported successfully!")

In [ ]:
# Step 1: Load the saved PMJDY state-wise dataset

# Load the cleaned dataset committed in this GitHub repository.
# This avoids depending on the PMJDY website's SSL certificate during every run.
DATA_URL = "https://raw.githubusercontent.com/NANCYSINGH6285/PMJDY-Financial-Inclusion/main/PMJDY_Statewise_Cleaned.csv"

try:
    df = pd.read_csv(DATA_URL)
    print("Dataset loaded from the project GitHub repository.")
except Exception as e:
    print("Could not download the project CSV:", e)
    print("Upload PMJDY_Statewise_Cleaned.csv in the Colab Files panel, then run this cell again.")
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv("PMJDY_Statewise_Cleaned.csv")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Step 2: Validate and clean the state-wise dataset

df.columns = df.columns.str.strip()
df["State Name"] = df["State Name"].astype(str).str.strip()

numeric_columns = [
    "Beneficiaries at rural/semi-urban centre bank branches",
    "Beneficiaries at urban/metro centre bank branches",
    "Total Beneficiaries",
    "Balance in beneficiary accounts (in crore)",
    "No. of RuPay cards issued to beneficiaries"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["State Name"] + numeric_columns)
df = df[~df["State Name"].str.lower().isin(["total", "grand total", "all india"])]
df = df.drop_duplicates(subset=["State Name"]).reset_index(drop=True)

calculated_total = (
    df["Beneficiaries at rural/semi-urban centre bank branches"]
    + df["Beneficiaries at urban/metro centre bank branches"]
)
mismatch_count = int((calculated_total != df["Total Beneficiaries"]).sum())

print("Rows after cleaning:", len(df))
print("Unique states/UTs:", df["State Name"].nunique())
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate state/UT names:", int(df["State Name"].duplicated().sum()))
print("Rural + urban total mismatches:", mismatch_count)

if mismatch_count:
    print("Warning: Some rural + urban totals do not match the reported total. Review source data.")

df.to_csv("PMJDY_Statewise_Cleaned.csv", index=False)
display(df.head())

In [ ]:

# Step 3: Create meaningful features for clustering

# Share of beneficiaries in rural/semi-urban centres
df["Rural Beneficiary Share (%)"] = (
    df["Beneficiaries at rural/semi-urban centre bank branches"]
    / df["Total Beneficiaries"] * 100
)

# Share of beneficiaries in urban/metro centres
df["Urban Beneficiary Share (%)"] = (
    df["Beneficiaries at urban/metro centre bank branches"]
    / df["Total Beneficiaries"] * 100
)

# Average account balance per beneficiary, in rupees
df["Average Balance per Beneficiary (Rs)"] = (
    df["Balance in beneficiary accounts (in crore)"]
    * 10_000_000
    / df["Total Beneficiaries"]
)

# RuPay cards issued per 100 beneficiaries
df["RuPay Cards per 100 Beneficiaries"] = (
    df["No. of RuPay cards issued to beneficiaries"]
    / df["Total Beneficiaries"] * 100
)

print("New features created successfully!")

display(df[[
    "State Name",
    "Total Beneficiaries",
    "Rural Beneficiary Share (%)",
    "Average Balance per Beneficiary (Rs)",
    "RuPay Cards per 100 Beneficiaries"
]].head(10))

print("\nFeature summary:")
display(df[[
    "Rural Beneficiary Share (%)",
    "Average Balance per Beneficiary (Rs)",
    "RuPay Cards per 100 Beneficiaries"
]].describe().round(2))

In [ ]:
# Step 4: Select features and scale the data

# Log transformation reduces the effect of very large beneficiary counts
df["Log Total Beneficiaries"] = np.log1p(df["Total Beneficiaries"])

# Features used for clustering
features = [
    "Log Total Beneficiaries",
    "Rural Beneficiary Share (%)",
    "Average Balance per Beneficiary (Rs)",
    "RuPay Cards per 100 Beneficiaries"
]

X = df[features].copy()

# Standardize all features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features selected:", features)
print("Original data shape:", X.shape)
print("Scaled data shape:", X_scaled.shape)
print("\nScaled data preview:")

display(pd.DataFrame(X_scaled, columns=features).head(10).round(2))

In [ ]:
# Step 5: Compare different numbers of clusters

inertia_values = []
silhouette_values = []
k_values = range(2, 7)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(X_scaled)

    inertia_values.append(kmeans.inertia_)
    silhouette_values.append(
        silhouette_score(X_scaled, labels)
    )

results = pd.DataFrame({
    "Number of Clusters (K)": list(k_values),
    "Inertia": inertia_values,
    "Silhouette Score": silhouette_values
})

display(results.round(3))

# Plot the Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertia_values, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for PMJDY State Clustering")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

# Plot Silhouette Scores
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), silhouette_values, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

In [ ]:
# Step 6: Perform K-Means clustering with K = 4

kmeans_final = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

df["KMeans_Cluster"] = kmeans_final.fit_predict(X_scaled)

print("K-Means clustering completed!")
print("\nNumber of states/UTs in each cluster:")

display(
    df["KMeans_Cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .to_frame("Number of States/UTs")
)

print("\nState-wise cluster assignments:")

display(
    df[[
        "State Name",
        "Total Beneficiaries",
        "Rural Beneficiary Share (%)",
        "Average Balance per Beneficiary (Rs)",
        "RuPay Cards per 100 Beneficiaries",
        "KMeans_Cluster"
    ]].sort_values("KMeans_Cluster")
)

In [ ]:
# Step 7: Visualize K-Means clusters using PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]

plt.figure(figsize=(10, 7))

for cluster in sorted(df["KMeans_Cluster"].unique()):
    cluster_data = df[df["KMeans_Cluster"] == cluster]

    plt.scatter(
        cluster_data["PCA1"],
        cluster_data["PCA2"],
        label=f"Cluster {cluster}",
        s=80
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PMJDY State-wise Financial Inclusion Clusters")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("PMJDY_KMeans_Clusters.png", dpi=300)
plt.show()

print(
    "Variance explained by two PCA components:",
    round(pca.explained_variance_ratio_.sum() * 100, 2),
    "%"
)

In [ ]:
# Step 8: Analyse the characteristics of each cluster

# Note: cluster IDs are arbitrary labels, not rankings.

cluster_profile = df.groupby("KMeans_Cluster").agg(
    Number_of_States=("State Name", "count"),
    Average_Beneficiaries=("Total Beneficiaries", "mean"),
    Average_Rural_Share=("Rural Beneficiary Share (%)", "mean"),
    Average_Balance_Rs=("Average Balance per Beneficiary (Rs)", "mean"),
    Average_RuPay_Indicator=("RuPay Cards per 100 Beneficiaries", "mean")
).round(2)

print("Cluster Profiles:")
display(cluster_profile)

# Save cluster assignments and profiles
df.to_csv("PMJDY_Clustered_States.csv", index=False)
cluster_profile.to_csv("PMJDY_Cluster_Profiles.csv")

print("Files saved successfully!")

In [ ]:
# Step 9: Final charts and project outputs

# Chart 1: Average rural beneficiary share by cluster
plt.figure(figsize=(8, 5))

cluster_profile["Average_Rural_Share"].plot(
    kind="bar",
    rot=0
)

plt.title("Average Rural Beneficiary Share by Cluster")
plt.xlabel("K-Means Cluster")
plt.ylabel("Rural Beneficiary Share (%)")
plt.tight_layout()
plt.savefig("PMJDY_Rural_Share_by_Cluster.png", dpi=300)
plt.show()


# Chart 2: Average balance per beneficiary by cluster
plt.figure(figsize=(8, 5))

cluster_profile["Average_Balance_Rs"].plot(
    kind="bar",
    rot=0
)

plt.title("Average Balance per Beneficiary by Cluster")
plt.xlabel("K-Means Cluster")
plt.ylabel("Average Balance (Rs)")
plt.tight_layout()
plt.savefig("PMJDY_Balance_by_Cluster.png", dpi=300)
plt.show()


# Save final dataset and cluster profile
df.to_csv("PMJDY_Clustered_States.csv", index=False)
cluster_profile.to_csv("PMJDY_Cluster_Profiles.csv")

print("Final charts and CSV files saved successfully!")
print("\nSaved files:")
print("1. PMJDY_Clustered_States.csv")
print("2. PMJDY_Cluster_Profiles.csv")
print("3. PMJDY_KMeans_Clusters.png")
print("4. PMJDY_Rural_Share_by_Cluster.png")
print("5. PMJDY_Balance_by_Cluster.png")

In [ ]:
# Step 10: Download the final project outputs (optional)

# Run this cell in Google Colab if you want to download the generated CSV files and charts.
from google.colab import files

for filename in [
    "PMJDY_Statewise_Cleaned.csv",
    "PMJDY_Clustered_States.csv",
    "PMJDY_Cluster_Profiles.csv",
    "PMJDY_KMeans_Clusters.png",
    "PMJDY_Rural_Share_by_Cluster.png",
    "PMJDY_Balance_by_Cluster.png"
]:
    files.download(filename)